# SN Helios: A Multiply Imaged Type II Supernova Opening Time-Delay Cosmography beyond redshift of 3

Paper: https://doi.org/10.48550/arXiv.2609.30440

---

# Part 1

In [ ]:
# SN HELIOS — A SUPERNOVA WE WILL SEE AGAIN
# Self-contained Jupyter cell
# Scientific schematic based on Fujimoto et al. (2026), arXiv:2609.30440
#
# Output: WebM (VP9) or MP4 (H.264)
# Duration: 22 s @ 30 fps
#
# The geometry is intentionally schematic and is NOT a reconstruction
# of the RXC J0018.5+1626 lens model.

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle, Ellipse, PathPatch
from matplotlib.path import Path
from matplotlib import patheffects

# ============================================================
# SETTINGS
# ============================================================

OUTPUT = "mp4"             # "webm" or "mp4"
FILENAME = "sn_helios_reappearance"

FPS = 30
DURATION = 22
NFRAMES = FPS * DURATION

FIGSIZE = (12.8, 7.2)
DPI = 120

# ============================================================
# HELPERS
# ============================================================

def clamp(x, a=0.0, b=1.0):
    return np.clip(x, a, b)

def smoothstep(x):
    x = clamp(x)
    return x * x * (3.0 - 2.0 * x)

def fade_window(t, t0, t1, fade=0.6):
    """Fade in at t0, fade out approaching t1."""
    if t < t0 or t > t1:
        return 0.0
    fi = smoothstep((t - t0) / fade)
    fo = smoothstep((t1 - t) / fade)
    return min(fi, fo)

def fade_in(t, t0, duration=0.7):
    return smoothstep((t - t0) / duration)

def set_text_alpha(txt, a):
    txt.set_alpha(clamp(a))

def bezier_points(p0, p1, p2, p3, n=240):
    u = np.linspace(0, 1, n)
    pts = ((1-u)**3)[:,None]*p0 \
        + (3*(1-u)**2*u)[:,None]*p1 \
        + (3*(1-u)*u**2)[:,None]*p2 \
        + (u**3)[:,None]*p3
    return pts

def moving_pulse(line, fraction, width=0.065):
    """
    Bright pulse travelling along a precomputed curve.
    Returns coordinates around the current pulse position.
    """
    fraction = clamp(fraction)
    n = len(line)
    center = int(fraction * (n - 1))
    half = max(2, int(width * n))
    lo = max(0, center - half)
    hi = min(n, center + half + 1)
    return line[lo:hi]

# ============================================================
# FIGURE
# ============================================================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor("#03070d")
ax.set_facecolor("#03070d")

ax.set_xlim(0, 16)
ax.set_ylim(0, 9)
ax.set_aspect("equal")
ax.axis("off")

# ============================================================
# STAR FIELD
# ============================================================

rng = np.random.default_rng(42)

sx = rng.uniform(0.1, 15.9, 240)
sy = rng.uniform(0.1, 8.9, 240)
ss = rng.uniform(1.0, 6.0, 240)

stars = ax.scatter(
    sx, sy,
    s=ss,
    c="white",
    alpha=0.0,
    linewidths=0,
    zorder=0
)

# ============================================================
# TITLE
# ============================================================

title = ax.text(
    0.65, 8.35,
    "SN HELIOS",
    fontsize=27,
    weight="bold",
    color="white",
    ha="left",
    va="center",
    alpha=0,
    zorder=20
)

subtitle = ax.text(
    0.67, 7.92,
    "A SUPERNOVA WE WILL SEE AGAIN",
    fontsize=13,
    color="#a9c9d8",
    ha="left",
    va="center",
    alpha=0,
    zorder=20
)

# ============================================================
# OBJECT POSITIONS
# ============================================================

SN = np.array([2.15, 4.55])
CL = np.array([8.0, 4.50])
OBS = np.array([14.35, 4.50])

# ============================================================
# SUPERNOVA
# ============================================================

sn_halo = Circle(
    SN, 0.32,
    facecolor="#dcefff",
    edgecolor="none",
    alpha=0,
    zorder=8
)
ax.add_patch(sn_halo)

sn_core = Circle(
    SN, 0.085,
    facecolor="white",
    edgecolor="none",
    alpha=0,
    zorder=9
)
ax.add_patch(sn_core)

sn_label = ax.text(
    SN[0], SN[1]-0.68,
    "SN HELIOS\nz = 3.34",
    fontsize=11,
    color="white",
    ha="center",
    va="top",
    linespacing=1.35,
    alpha=0,
    zorder=20
)

# ============================================================
# CLUSTER / GRAVITATIONAL LENS
# ============================================================

cluster_glow = Ellipse(
    CL, 2.15, 2.65,
    facecolor="#173248",
    edgecolor="#5f93aa",
    linewidth=1.0,
    alpha=0,
    zorder=2
)
ax.add_patch(cluster_glow)

# cluster galaxies
galaxy_offsets = np.array([
    [ 0.00,  0.00],
    [-0.55,  0.43],
    [ 0.62,  0.35],
    [-0.62, -0.42],
    [ 0.52, -0.50],
    [ 0.10,  0.82],
    [-0.12, -0.87],
])

cluster_gals = []
for i, off in enumerate(galaxy_offsets):
    e = Ellipse(
        CL + off,
        0.36 if i else 0.52,
        0.18 if i else 0.25,
        angle=(i * 31) % 160,
        facecolor="#d6e4e8",
        edgecolor="none",
        alpha=0,
        zorder=5
    )
    ax.add_patch(e)
    cluster_gals.append(e)

cluster_label = ax.text(
    CL[0], 2.78,
    "RXC J0018.5+1626\nGRAVITATIONAL LENS",
    fontsize=10.5,
    color="#a9c9d8",
    ha="center",
    va="top",
    linespacing=1.35,
    alpha=0,
    zorder=20
)

# ============================================================
# OBSERVER / JWST SYMBOL
# ============================================================

observer_outer = Circle(
    OBS, 0.24,
    facecolor="none",
    edgecolor="#9fc6d7",
    linewidth=1.5,
    alpha=0,
    zorder=8
)
observer_inner = Circle(
    OBS, 0.065,
    facecolor="white",
    edgecolor="none",
    alpha=0,
    zorder=9
)

ax.add_patch(observer_outer)
ax.add_patch(observer_inner)

observer_label = ax.text(
    OBS[0], OBS[1]-0.55,
    "OBSERVER",
    fontsize=10.5,
    color="#a9c9d8",
    ha="center",
    va="top",
    alpha=0,
    zorder=20
)

# ============================================================
# LIGHT PATHS
# ============================================================

upper = bezier_points(
    SN,
    np.array([5.3, 5.1]),
    np.array([9.2, 6.55]),
    OBS
)

middle = bezier_points(
    SN,
    np.array([5.4, 4.7]),
    np.array([10.0, 4.95]),
    OBS
)

lower = bezier_points(
    SN,
    np.array([5.3, 3.9]),
    np.array([9.2, 2.35]),
    OBS
)

paths = [upper, middle, lower]

path_lines = []
pulse_lines = []

for pts in paths:
    base, = ax.plot(
        pts[:,0], pts[:,1],
        color="#6cb4cf",
        lw=1.2,
        alpha=0,
        zorder=4
    )
    pulse, = ax.plot(
        [], [],
        color="white",
        lw=3.0,
        alpha=0,
        solid_capstyle="round",
        zorder=7
    )
    pulse.set_path_effects([
        patheffects.Stroke(linewidth=6, foreground="#68b9d6", alpha=0.20),
        patheffects.Normal()
    ])

    path_lines.append(base)
    pulse_lines.append(pulse)

# ============================================================
# STORY TEXT
# ============================================================

explosion_text = ax.text(
    0.72, 1.20,
    "ONE EXPLOSION",
    fontsize=18,
    weight="bold",
    color="white",
    ha="left",
    alpha=0,
    zorder=20
)

lens_text = ax.text(
    0.72, 0.72,
    "GRAVITY SENDS ITS LIGHT ALONG DIFFERENT PATHS",
    fontsize=11,
    color="#9fc6d7",
    ha="left",
    alpha=0,
    zorder=20
)

arrival1 = ax.text(
    12.2, 6.55,
    "IMAGE ARRIVES",
    fontsize=11,
    color="white",
    ha="center",
    alpha=0,
    zorder=20
)

arrival2 = ax.text(
    12.15, 2.00,
    "ANOTHER IMAGE\nIS STILL ON ITS WAY",
    fontsize=11,
    color="#a9c9d8",
    ha="center",
    linespacing=1.35,
    alpha=0,
    zorder=20
)

# ============================================================
# FINAL CARD
# ============================================================

final_overlay = plt.Rectangle(
    (0, 0), 16, 9,
    facecolor="#03070d",
    edgecolor="none",
    alpha=0,
    zorder=50
)
ax.add_patch(final_overlay)

final_title = ax.text(
    8, 7.15,
    "SN HELIOS",
    fontsize=32,
    weight="bold",
    color="white",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_type = ax.text(
    8, 6.35,
    "TYPE II SUPERNOVA   •   z = 3.34",
    fontsize=15,
    color="#a9c9d8",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_statement = ax.text(
    8, 4.95,
    "ONE EXPLOSION.\nMULTIPLE IMAGES.",
    fontsize=22,
    weight="bold",
    color="white",
    ha="center",
    va="center",
    linespacing=1.45,
    alpha=0,
    zorder=60
)

final_next = ax.text(
    8, 3.15,
    "NEXT APPEARANCE",
    fontsize=14,
    color="#8eacba",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_years = ax.text(
    8, 2.38,
    "≈ 2–6 YEARS",
    fontsize=30,
    weight="bold",
    color="#dff5ff",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_note = ax.text(
    8, 1.35,
    "Predicted by independent gravitational-lens models",
    fontsize=9.5,
    color="#728894",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

# ============================================================
# CLOCK — lower left
# ============================================================

clock_center = np.array([0.68, 0.58])
clock_r = 0.29

clock_circle = Circle(
    clock_center,
    clock_r,
    facecolor="none",
    edgecolor="#5e7681",
    linewidth=0.8,
    alpha=0.6,
    zorder=70
)
ax.add_patch(clock_circle)

clock_ticks = []
for i in range(12):
    ang = 2*np.pi*i/12
    p1 = clock_center + 0.21*np.array([np.sin(ang), np.cos(ang)])
    p2 = clock_center + 0.255*np.array([np.sin(ang), np.cos(ang)])
    ln, = ax.plot(
        [p1[0], p2[0]],
        [p1[1], p2[1]],
        color="#5e7681",
        lw=0.7,
        alpha=0.6,
        zorder=70
    )
    clock_ticks.append(ln)

clock_hand, = ax.plot(
    [], [],
    color="#b8d5df",
    lw=1.2,
    alpha=0.8,
    zorder=71
)

# ============================================================
# ANIMATION
# ============================================================

def update(frame):

    t = frame / FPS

    # --------------------------------------------------------
    # CLOCK — one revolution over entire animation
    # --------------------------------------------------------

    theta = 2*np.pi * t / DURATION
    hand_end = clock_center + 0.205 * np.array([
        np.sin(theta),
        np.cos(theta)
    ])

    clock_hand.set_data(
        [clock_center[0], hand_end[0]],
        [clock_center[1], hand_end[1]]
    )

    # --------------------------------------------------------
    # BACKGROUND
    # --------------------------------------------------------

    stars.set_alpha(0.20 * fade_in(t, 0.0, 1.0))

    set_text_alpha(title, fade_in(t, 0.2, 0.8))
    set_text_alpha(subtitle, fade_in(t, 0.7, 0.8))

    # --------------------------------------------------------
    # 0.8–4.0 s : SUPERNOVA
    # --------------------------------------------------------

    sn_a = fade_in(t, 1.0, 0.8)

    # subtle breathing glow
    pulse = 1.0 + 0.08*np.sin(t*5.0)
    sn_halo.set_radius(0.32 * pulse)

    sn_halo.set_alpha(0.22 * sn_a)
    sn_core.set_alpha(0.95 * sn_a)
    set_text_alpha(sn_label, fade_in(t, 1.6, 0.7))

    set_text_alpha(
        explosion_text,
        fade_window(t, 2.1, 7.0, 0.7)
    )

    # --------------------------------------------------------
    # 3–7 s : CLUSTER APPEARS
    # --------------------------------------------------------

    cl_a = fade_in(t, 3.0, 0.9)

    cluster_glow.set_alpha(0.18 * cl_a)

    for i, g in enumerate(cluster_gals):
        g.set_alpha((0.55 if i else 0.85) * cl_a)

    set_text_alpha(
        cluster_label,
        fade_in(t, 3.7, 0.8)
    )

    set_text_alpha(
        lens_text,
        fade_window(t, 4.2, 10.5, 0.8)
    )

    # --------------------------------------------------------
    # OBSERVER
    # --------------------------------------------------------

    obs_a = fade_in(t, 4.5, 0.8)

    observer_outer.set_alpha(0.8 * obs_a)
    observer_inner.set_alpha(0.9 * obs_a)
    set_text_alpha(observer_label, obs_a)

    # --------------------------------------------------------
    # PATHS
    # --------------------------------------------------------

    path_a = fade_in(t, 5.0, 1.0)

    for i, line in enumerate(path_lines):
        line.set_alpha((0.38 if i != 1 else 0.55) * path_a)

    # --------------------------------------------------------
    # travelling photons
    # Path 1 reaches observer first
    # --------------------------------------------------------

    timings = [
        (5.6, 9.2),
        (6.4, 11.3),
        (7.0, 14.7)
    ]

    for i, (ts, te) in enumerate(timings):

        frac = (t - ts) / (te - ts)

        if 0 <= frac <= 1:
            pp = moving_pulse(paths[i], frac)
            pulse_lines[i].set_data(pp[:,0], pp[:,1])
            pulse_lines[i].set_alpha(0.95)
        else:
            pulse_lines[i].set_data([], [])
            pulse_lines[i].set_alpha(0)

    # first arrival
    set_text_alpha(
        arrival1,
        fade_window(t, 9.0, 12.3, 0.6)
    )

    # delayed path
    set_text_alpha(
        arrival2,
        fade_window(t, 11.7, 15.5, 0.7)
    )

    # --------------------------------------------------------
    # 14.7–16.0 s : hold the delayed-light idea
    # --------------------------------------------------------

    if 14.7 <= t < 16.0:
        # keep lower path subtly highlighted
        path_lines[2].set_alpha(0.80)

    # --------------------------------------------------------
    # FINAL CARD starts at 16 s
    # Gives ~6 seconds of stable reading time.
    # --------------------------------------------------------

    fa = smoothstep((t - 15.7) / 0.7)
    final_overlay.set_alpha(0.965 * fa)

    set_text_alpha(final_title, fade_in(t, 16.0, 0.45))
    set_text_alpha(final_type, fade_in(t, 16.25, 0.45))
    set_text_alpha(final_statement, fade_in(t, 16.55, 0.55))
    set_text_alpha(final_next, fade_in(t, 16.9, 0.45))
    set_text_alpha(final_years, fade_in(t, 17.15, 0.55))
    set_text_alpha(final_note, fade_in(t, 17.45, 0.55))

    return []

# ============================================================
# RENDER
# ============================================================

ani = animation.FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000/FPS,
    blit=False
)

plt.tight_layout(pad=0)

if OUTPUT.lower() == "webm":

    writer = animation.FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-deadline", "good",
            "-cpu-used", "2"
        ]
    )

    outfile = FILENAME + ".webm"

else:

    writer = animation.FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-preset", "medium"
        ]
    )

    outfile = FILENAME + ".mp4"

ani.save(outfile, writer=writer, dpi=DPI)

plt.close(fig)

print(f"Saved: {outfile}")

SN Helios — сверхновая II типа на красном смещении z = 3,34. Её свет проходит через гравитационное поле массивного скопления галактик RXC J0018.5+1626 несколькими путями различной длины. Поэтому один и тот же взрыв наблюдается в разные моменты времени. Независимые модели гравитационной линзы предсказывают появление следующего изображения сверхновой примерно через 2–6 лет. 

SN Helios is a Type II supernova at redshift z = 3.34. Its light is gravitationally lensed by the massive galaxy cluster RXC J0018.5+1626 and reaches us along paths with different travel times. As a result, the same stellar explosion can appear more than once. Independent lens models predict another image of SN Helios in approximately 2–6 years

---

# Part 2. GRAVITY CREATES A COSMIC REPLAY

In [ ]:
# GRAVITY CREATES A COSMIC REPLAY
# SN Helios — gravitational-lens time delay
#
# Self-contained Jupyter cell
# Scientific schematic based on Fujimoto et al. (2026), arXiv:2609.30440
#
# Output: WebM (VP9) or MP4 (H.264)
# Duration: 22 s @ 30 fps
#
# IMPORTANT:
# This is a conceptual gravitational-lensing diagram.
# The trajectories are schematic and are NOT reconstructed
# light paths from a specific RXC J0018.5+1626 lens model.

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle, Ellipse, FancyArrowPatch
from matplotlib import patheffects

# ============================================================
# SETTINGS
# ============================================================

OUTPUT = "mp4"       # "webm" or "mp4"
FILENAME = "sn_helios_cosmic_replay"

FPS = 30
DURATION = 22
NFRAMES = FPS * DURATION

FIGSIZE = (12.8, 7.2)
DPI = 120

# ============================================================
# HELPERS
# ============================================================

def clamp(x, a=0.0, b=1.0):
    return np.clip(x, a, b)

def smoothstep(x):
    x = clamp(x)
    return x*x*(3.0 - 2.0*x)

def fade_in(t, t0, duration=0.7):
    return smoothstep((t-t0)/duration)

def fade_out(t, t0, duration=0.7):
    return 1.0 - smoothstep((t-t0)/duration)

def fade_window(t, t0, t1, fade=0.6):
    if t < t0 or t > t1:
        return 0.0
    return min(
        smoothstep((t-t0)/fade),
        smoothstep((t1-t)/fade)
    )

def set_alpha(obj, a):
    obj.set_alpha(clamp(a))

def bezier(p0, p1, p2, p3, n=400):
    u = np.linspace(0, 1, n)
    return (
        ((1-u)**3)[:,None]*p0 +
        (3*(1-u)**2*u)[:,None]*p1 +
        (3*(1-u)*u**2)[:,None]*p2 +
        (u**3)[:,None]*p3
    )

def pulse_segment(points, fraction, width=0.035):
    fraction = clamp(fraction)

    n = len(points)
    center = int(fraction*(n-1))
    half = max(3, int(width*n))

    lo = max(0, center-half)
    hi = min(n, center+half+1)

    return points[lo:hi]

# ============================================================
# FIGURE
# ============================================================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

fig.patch.set_facecolor("#03070d")
ax.set_facecolor("#03070d")

ax.set_xlim(0, 16)
ax.set_ylim(0, 9)
ax.set_aspect("equal")
ax.axis("off")

# ============================================================
# BACKGROUND STARS
# ============================================================

rng = np.random.default_rng(17)

sx = rng.uniform(0.15, 15.85, 260)
sy = rng.uniform(0.15, 8.85, 260)
ss = rng.uniform(0.7, 5.0, 260)

stars = ax.scatter(
    sx, sy,
    s=ss,
    c="white",
    alpha=0,
    linewidths=0,
    zorder=0
)

# ============================================================
# TITLE
# ============================================================

title = ax.text(
    0.65, 8.35,
    "GRAVITY CREATES A COSMIC REPLAY",
    fontsize=24,
    weight="bold",
    color="white",
    ha="left",
    va="center",
    alpha=0,
    zorder=30
)

subtitle = ax.text(
    0.67, 7.92,
    "THE SAME EXPLOSION CAN REACH US MORE THAN ONCE",
    fontsize=11.5,
    color="#9fc6d7",
    ha="left",
    va="center",
    alpha=0,
    zorder=30
)

# ============================================================
# POSITIONS
# ============================================================

SN = np.array([2.0, 4.5])
LENS = np.array([8.0, 4.5])
OBS = np.array([14.3, 4.5])

# ============================================================
# SUPERNOVA
# ============================================================

sn_outer = Circle(
    SN, 0.32,
    facecolor="#bde7ff",
    edgecolor="none",
    alpha=0,
    zorder=10
)

sn_inner = Circle(
    SN, 0.08,
    facecolor="white",
    edgecolor="none",
    alpha=0,
    zorder=11
)

ax.add_patch(sn_outer)
ax.add_patch(sn_inner)

sn_label = ax.text(
    SN[0], 3.88,
    "SN HELIOS\nz = 3.34",
    fontsize=10.5,
    color="white",
    ha="center",
    va="top",
    linespacing=1.3,
    alpha=0,
    zorder=30
)

# ============================================================
# CLUSTER / LENS
# ============================================================

lens_halo1 = Ellipse(
    LENS,
    2.35, 3.0,
    facecolor="#163246",
    edgecolor="#51849a",
    linewidth=0.9,
    alpha=0,
    zorder=2
)

lens_halo2 = Ellipse(
    LENS,
    3.15, 3.9,
    facecolor="none",
    edgecolor="#355c6d",
    linewidth=0.7,
    alpha=0,
    zorder=1
)

ax.add_patch(lens_halo1)
ax.add_patch(lens_halo2)

galaxy_offsets = np.array([
    [ 0.00,  0.00],
    [-0.58,  0.52],
    [ 0.62,  0.42],
    [-0.68, -0.42],
    [ 0.57, -0.56],
    [ 0.08,  0.92],
    [-0.08, -0.95],
    [-0.88,  0.05],
    [ 0.91, -0.02],
])

galaxies = []

for i, off in enumerate(galaxy_offsets):

    g = Ellipse(
        LENS + off,
        0.50 if i == 0 else 0.30,
        0.24 if i == 0 else 0.14,
        angle=(i*37) % 170,
        facecolor="#d7e4e8",
        edgecolor="none",
        alpha=0,
        zorder=6
    )

    ax.add_patch(g)
    galaxies.append(g)

lens_label = ax.text(
    LENS[0], 2.52,
    "GALAXY CLUSTER\nGRAVITATIONAL LENS",
    fontsize=10.5,
    color="#9fc6d7",
    ha="center",
    va="top",
    linespacing=1.35,
    alpha=0,
    zorder=30
)

# ============================================================
# OBSERVER
# ============================================================

obs_outer = Circle(
    OBS, 0.25,
    facecolor="none",
    edgecolor="#9fc6d7",
    linewidth=1.4,
    alpha=0,
    zorder=10
)

obs_inner = Circle(
    OBS, 0.06,
    facecolor="white",
    edgecolor="none",
    alpha=0,
    zorder=11
)

ax.add_patch(obs_outer)
ax.add_patch(obs_inner)

obs_label = ax.text(
    OBS[0], 3.95,
    "OBSERVER",
    fontsize=10,
    color="#9fc6d7",
    ha="center",
    va="top",
    alpha=0,
    zorder=30
)

# ============================================================
# LIGHT PATHS
# ============================================================

# Three intentionally different path lengths / shapes.

path_fast = bezier(
    SN,
    np.array([5.1, 5.00]),
    np.array([10.1, 5.05]),
    OBS
)

path_mid = bezier(
    SN,
    np.array([5.0, 5.55]),
    np.array([9.4, 6.35]),
    OBS
)

path_slow = bezier(
    SN,
    np.array([4.9, 3.35]),
    np.array([9.4, 2.15]),
    OBS
)

paths = [path_fast, path_mid, path_slow]

base_lines = []
pulse_lines = []

for pts in paths:

    line, = ax.plot(
        pts[:,0], pts[:,1],
        color="#63aac4",
        lw=1.2,
        alpha=0,
        zorder=4
    )

    pulse, = ax.plot(
        [], [],
        color="white",
        lw=3.2,
        alpha=0,
        solid_capstyle="round",
        zorder=9
    )

    pulse.set_path_effects([
        patheffects.Stroke(
            linewidth=7,
            foreground="#67bedc",
            alpha=0.18
        ),
        patheffects.Normal()
    ])

    base_lines.append(line)
    pulse_lines.append(pulse)

# ============================================================
# CENTRAL EXPLANATION
# ============================================================

one_event = ax.text(
    0.72, 1.18,
    "ONE EXPLOSION",
    fontsize=18,
    weight="bold",
    color="white",
    ha="left",
    alpha=0,
    zorder=30
)

different_paths = ax.text(
    0.72, 0.70,
    "DIFFERENT LIGHT PATHS  →  DIFFERENT ARRIVAL TIMES",
    fontsize=11,
    color="#9fc6d7",
    ha="left",
    alpha=0,
    zorder=30
)

# ============================================================
# ARRIVAL MARKERS
# ============================================================

arrival_fast = ax.text(
    13.05, 5.48,
    "ARRIVES FIRST",
    fontsize=10,
    weight="bold",
    color="white",
    ha="center",
    alpha=0,
    zorder=30
)

arrival_mid = ax.text(
    11.75, 6.65,
    "DELAYED",
    fontsize=10,
    color="#b8d5df",
    ha="center",
    alpha=0,
    zorder=30
)

arrival_slow = ax.text(
    11.70, 1.78,
    "DELAYED EVEN MORE",
    fontsize=10,
    color="#b8d5df",
    ha="center",
    alpha=0,
    zorder=30
)

# ============================================================
# TIME-DELAY EXPLANATION
# ============================================================

delta_title = ax.text(
    8.0, 1.18,
    "TIME DELAY",
    fontsize=13,
    weight="bold",
    color="white",
    ha="center",
    alpha=0,
    zorder=30
)

delta_text = ax.text(
    8.0, 0.72,
    "Δt  =  DIFFERENCE IN LIGHT-TRAVEL TIME",
    fontsize=11,
    color="#9fc6d7",
    ha="center",
    alpha=0,
    zorder=30
)

# ============================================================
# FINAL CARD
# ============================================================

final_overlay = plt.Rectangle(
    (0,0), 16,9,
    facecolor="#03070d",
    edgecolor="none",
    alpha=0,
    zorder=50
)

ax.add_patch(final_overlay)

final_title = ax.text(
    8, 7.0,
    "GRAVITY CREATES A COSMIC REPLAY",
    fontsize=25,
    weight="bold",
    color="white",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_one = ax.text(
    8, 5.65,
    "ONE EXPLOSION",
    fontsize=18,
    color="#a9c9d8",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_paths = ax.text(
    8, 4.65,
    "MULTIPLE LIGHT PATHS",
    fontsize=23,
    weight="bold",
    color="white",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_arrow = ax.text(
    8, 3.75,
    "↓",
    fontsize=22,
    color="#7999a7",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_delay = ax.text(
    8, 2.85,
    "DIFFERENT ARRIVAL TIMES",
    fontsize=23,
    weight="bold",
    color="#dff5ff",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_helios = ax.text(
    8, 1.65,
    "SN HELIOS: another image is predicted in ≈ 2–6 years",
    fontsize=11,
    color="#8faab6",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

# ============================================================
# CLOCK
# ============================================================

clock_center = np.array([0.68, 0.58])
clock_r = 0.29

clock_circle = Circle(
    clock_center,
    clock_r,
    facecolor="none",
    edgecolor="#5e7681",
    linewidth=0.8,
    alpha=0.6,
    zorder=70
)

ax.add_patch(clock_circle)

for i in range(12):

    ang = 2*np.pi*i/12

    p1 = clock_center + 0.21*np.array([
        np.sin(ang), np.cos(ang)
    ])

    p2 = clock_center + 0.255*np.array([
        np.sin(ang), np.cos(ang)
    ])

    ax.plot(
        [p1[0], p2[0]],
        [p1[1], p2[1]],
        color="#5e7681",
        lw=0.7,
        alpha=0.6,
        zorder=70
    )

clock_hand, = ax.plot(
    [], [],
    color="#b8d5df",
    lw=1.2,
    alpha=0.8,
    zorder=71
)

# ============================================================
# ANIMATION
# ============================================================

def update(frame):

    t = frame / FPS

    # --------------------------------------------------------
    # CLOCK
    # --------------------------------------------------------

    theta = 2*np.pi*t/DURATION

    hand_end = clock_center + 0.205*np.array([
        np.sin(theta),
        np.cos(theta)
    ])

    clock_hand.set_data(
        [clock_center[0], hand_end[0]],
        [clock_center[1], hand_end[1]]
    )

    # --------------------------------------------------------
    # BACKGROUND + TITLE
    # --------------------------------------------------------

    stars.set_alpha(0.18 * fade_in(t, 0.0, 1.0))

    set_alpha(title, fade_in(t, 0.2, 0.8))
    set_alpha(subtitle, fade_in(t, 0.8, 0.8))

    # --------------------------------------------------------
    # SN
    # --------------------------------------------------------

    sa = fade_in(t, 1.3, 0.7)

    # explosion pulse early, then settle
    if t < 3.0:
        glow_scale = 1.0 + 0.55*np.exp(-((t-1.8)/0.45)**2)
    else:
        glow_scale = 1.0 + 0.05*np.sin(t*4)

    sn_outer.set_radius(0.32 * glow_scale)
    sn_outer.set_alpha(0.24 * sa)
    sn_inner.set_alpha(0.95 * sa)

    set_alpha(sn_label, fade_in(t, 1.8, 0.6))

    set_alpha(
        one_event,
        fade_window(t, 2.0, 8.2, 0.7)
    )

    # --------------------------------------------------------
    # LENS
    # --------------------------------------------------------

    la = fade_in(t, 3.0, 0.8)

    lens_halo1.set_alpha(0.17*la)
    lens_halo2.set_alpha(0.30*la)

    for i,g in enumerate(galaxies):
        g.set_alpha((0.82 if i == 0 else 0.50)*la)

    set_alpha(lens_label, fade_in(t, 3.6, 0.7))

    # --------------------------------------------------------
    # OBSERVER
    # --------------------------------------------------------

    oa = fade_in(t, 4.0, 0.7)

    obs_outer.set_alpha(0.85*oa)
    obs_inner.set_alpha(0.95*oa)
    set_alpha(obs_label, oa)

    # --------------------------------------------------------
    # LIGHT PATHS APPEAR
    # --------------------------------------------------------

    pa = fade_in(t, 4.5, 0.8)

    for line in base_lines:
        line.set_alpha(0.35*pa)

    set_alpha(
        different_paths,
        fade_window(t, 4.8, 10.2, 0.7)
    )

    # --------------------------------------------------------
    # PHOTON PULSES
    #
    # All photons are emitted at the same conceptual event.
    # Their screen travel times differ to illustrate Δt.
    # --------------------------------------------------------

    timings = [
        (5.2, 8.4),
        (5.2, 10.5),
        (5.2, 13.2)
    ]

    for i, (ts,te) in enumerate(timings):

        frac = (t-ts)/(te-ts)

        if 0 <= frac <= 1:

            seg = pulse_segment(paths[i], frac)

            pulse_lines[i].set_data(
                seg[:,0],
                seg[:,1]
            )

            pulse_lines[i].set_alpha(0.98)

        else:

            pulse_lines[i].set_data([],[])
            pulse_lines[i].set_alpha(0)

    # --------------------------------------------------------
    # ARRIVALS
    # --------------------------------------------------------

    set_alpha(
        arrival_fast,
        fade_window(t, 8.2, 10.2, 0.4)
    )

    set_alpha(
        arrival_mid,
        fade_window(t, 10.3, 12.0, 0.4)
    )

    set_alpha(
        arrival_slow,
        fade_window(t, 13.0, 15.0, 0.4)
    )

    # highlight each path once it has demonstrated its delay

    if 8.4 <= t < 10.4:
        base_lines[0].set_alpha(0.80)

    if 10.5 <= t < 12.4:
        base_lines[1].set_alpha(0.80)

    if 13.2 <= t < 15.4:
        base_lines[2].set_alpha(0.80)

    # --------------------------------------------------------
    # TIME DELAY
    # --------------------------------------------------------

    set_alpha(
        delta_title,
        fade_window(t, 11.2, 15.5, 0.6)
    )

    set_alpha(
        delta_text,
        fade_window(t, 11.7, 15.5, 0.6)
    )

    # --------------------------------------------------------
    # FINAL CARD
    # Begins at ~15.7 s.
    # Stable for roughly six seconds.
    # --------------------------------------------------------

    final_a = smoothstep((t-15.5)/0.7)
    final_overlay.set_alpha(0.965*final_a)

    set_alpha(
        final_title,
        fade_in(t, 15.8, 0.45)
    )

    set_alpha(
        final_one,
        fade_in(t, 16.1, 0.45)
    )

    set_alpha(
        final_paths,
        fade_in(t, 16.35, 0.45)
    )

    set_alpha(
        final_arrow,
        fade_in(t, 16.55, 0.4)
    )

    set_alpha(
        final_delay,
        fade_in(t, 16.75, 0.45)
    )

    set_alpha(
        final_helios,
        fade_in(t, 17.05, 0.5)
    )

    return []

# ============================================================
# RENDER
# ============================================================

ani = animation.FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000/FPS,
    blit=False
)

plt.tight_layout(pad=0)

if OUTPUT.lower() == "webm":

    writer = animation.FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-deadline", "good",
            "-cpu-used", "2"
        ]
    )

    outfile = FILENAME + ".webm"

else:

    writer = animation.FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-preset", "medium"
        ]
    )

    outfile = FILENAME + ".mp4"

ani.save(outfile, writer=writer, dpi=DPI)

plt.close(fig)

print(f"Saved: {outfile}")

Массивное скопление галактик искривляет пространство-время и действует как гравитационная линза. Свет одного и того же взрыва SN Helios может пройти вокруг линзы различными путями. Поскольку эти пути имеют различные времена распространения, изображения одной сверхновой достигают наблюдателя в разные моменты. Так возникает гравитационная временная задержка. Для SN Helios следующее изображение предсказывается примерно через 2–6 лет.

A massive galaxy cluster curves spacetime and acts as a gravitational lens. Light from the same SN Helios explosion can travel around the lens along different paths. Because these paths have different travel times, images of the same supernova reach the observer at different epochs. This produces a gravitational time delay. For SN Helios, another image is predicted to appear in approximately 2–6 years.

---

# Part 3. THE SPECTRUM REVEALS THE EXPLOSION

In [ ]:
# THE SPECTRUM REVEALS THE EXPLOSION
# SN Helios — spectroscopy identifies a Type II supernova at z = 3.34
#
# Self-contained Jupyter cell
# Scientific schematic based on Fujimoto et al. (2026), arXiv:2609.30440
#
# Output: WebM (VP9) or MP4 (H.264)
# Duration: 23 s @ 30 fps
#
# NOTE:
# The spectrum drawn here is a conceptual visualization of the
# reported spectral features, NOT a digitization of the paper spectrum.

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle

# ============================================================
# SETTINGS
# ============================================================

OUTPUT = "mp4"       # "webm" or "mp4"
FILENAME = "sn_helios_spectrum"

FPS = 30
DURATION = 23
NFRAMES = FPS * DURATION

FIGSIZE = (12.8, 7.2)
DPI = 120

# ============================================================
# HELPERS
# ============================================================

def clamp(x, a=0.0, b=1.0):
    return np.clip(x, a, b)

def smoothstep(x):
    x = clamp(x)
    return x*x*(3.0 - 2.0*x)

def fade_in(t, t0, duration=0.7):
    return smoothstep((t-t0)/duration)

def fade_window(t, t0, t1, fade=0.6):
    if t < t0 or t > t1:
        return 0.0
    return min(
        smoothstep((t-t0)/fade),
        smoothstep((t1-t)/fade)
    )

def set_alpha(obj, a):
    obj.set_alpha(clamp(a))

def gaussian(x, mu, sigma):
    return np.exp(-0.5*((x-mu)/sigma)**2)

# ============================================================
# FIGURE
# ============================================================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

fig.patch.set_facecolor("#03070d")
ax.set_facecolor("#03070d")

ax.set_xlim(0, 16)
ax.set_ylim(0, 9)
ax.axis("off")

# ============================================================
# BACKGROUND STARS
# ============================================================

rng = np.random.default_rng(31)

sx = rng.uniform(0.1, 15.9, 230)
sy = rng.uniform(0.1, 8.9, 230)
ss = rng.uniform(0.8, 5.0, 230)

stars = ax.scatter(
    sx, sy,
    s=ss,
    c="white",
    alpha=0,
    linewidths=0,
    zorder=0
)

# ============================================================
# TITLE
# ============================================================

title = ax.text(
    0.65, 8.35,
    "THE SPECTRUM REVEALS THE EXPLOSION",
    fontsize=24,
    weight="bold",
    color="white",
    ha="left",
    va="center",
    alpha=0,
    zorder=40
)

subtitle = ax.text(
    0.67, 7.92,
    "SN HELIOS  •  JWST / NIRSpec",
    fontsize=11.5,
    color="#9fc6d7",
    ha="left",
    va="center",
    alpha=0,
    zorder=40
)

# ============================================================
# DISCOVERY SOURCE
# ============================================================

source_xy = np.array([8.0, 4.65])

source_halo = Circle(
    source_xy,
    0.38,
    facecolor="#d5b7a5",
    edgecolor="none",
    alpha=0,
    zorder=8
)

source_core = Circle(
    source_xy,
    0.075,
    facecolor="#fff4e9",
    edgecolor="none",
    alpha=0,
    zorder=9
)

ax.add_patch(source_halo)
ax.add_patch(source_core)

source_label = ax.text(
    8.0, 3.95,
    "A FAINT RED TRANSIENT",
    fontsize=15,
    weight="bold",
    color="white",
    ha="center",
    va="center",
    alpha=0,
    zorder=20
)

source_info = ax.text(
    8.0, 3.48,
    "F277W = 26.1 mag",
    fontsize=11,
    color="#9fc6d7",
    ha="center",
    va="center",
    alpha=0,
    zorder=20
)

question = ax.text(
    8.0, 2.35,
    "WHAT EXPLODED?",
    fontsize=20,
    weight="bold",
    color="white",
    ha="center",
    va="center",
    alpha=0,
    zorder=20
)

# ============================================================
# SPECTRUM COORDINATES
# ============================================================

x0 = 1.35
x1 = 14.75

y0 = 2.0
y1 = 6.65

# Normalized spectral coordinate.
# Labels correspond to the rest-frame features discussed in the paper.

wave = np.linspace(0, 1, 900)

# continuum
continuum = (
    0.38
    + 0.055*np.sin(wave*7.0)
    + 0.035*np.cos(wave*17.0)
)

# H-beta region
hbeta_x = 0.30
hbeta = (
    0.26*gaussian(wave, hbeta_x+0.018, 0.020)
    - 0.10*gaussian(wave, hbeta_x-0.022, 0.014)
)

# H-alpha: strong broad P-Cygni-like schematic profile
halpha_x = 0.60

halpha = (
    0.66*gaussian(wave, halpha_x+0.020, 0.034)
    - 0.28*gaussian(wave, halpha_x-0.035, 0.022)
)

# He I 10830
hei_x = 0.86
hei = (
    0.27*gaussian(wave, hei_x+0.012, 0.025)
    - 0.09*gaussian(wave, hei_x-0.025, 0.018)
)

flux = continuum + hbeta + halpha + hei

# map into plotting area

spec_x = x0 + wave*(x1-x0)

fmin = 0.12
fmax = 1.15

spec_y = y0 + (flux-fmin)/(fmax-fmin)*(y1-y0)

# ============================================================
# AXES
# ============================================================

axis_x, = ax.plot(
    [x0, x1],
    [y0, y0],
    color="#708894",
    lw=0.8,
    alpha=0,
    zorder=4
)

axis_y, = ax.plot(
    [x0, x0],
    [y0, y1],
    color="#708894",
    lw=0.8,
    alpha=0,
    zorder=4
)

xlabel = ax.text(
    (x0+x1)/2, 1.48,
    "WAVELENGTH",
    fontsize=10,
    color="#8199a4",
    ha="center",
    alpha=0,
    zorder=20
)

ylabel = ax.text(
    0.72, 4.3,
    "FLUX",
    fontsize=10,
    color="#8199a4",
    rotation=90,
    ha="center",
    va="center",
    alpha=0,
    zorder=20
)

# ============================================================
# SPECTRUM LINE
# ============================================================

spectrum_line, = ax.plot(
    [],
    [],
    color="#dff5ff",
    lw=1.8,
    alpha=0,
    zorder=10
)

# faint underlying complete spectrum
spectrum_ghost, = ax.plot(
    spec_x,
    spec_y,
    color="#528095",
    lw=0.8,
    alpha=0,
    zorder=5
)

# ============================================================
# FEATURE MARKERS
# ============================================================

def spectrum_y_at(position):
    idx = np.argmin(np.abs(wave-position))
    return spec_y[idx]

feature_data = [
    ("Hβ", hbeta_x+0.018),
    ("Hα", halpha_x+0.020),
    ("He I", hei_x+0.012)
]

feature_lines = []
feature_labels = []

for label, pos in feature_data:

    xx = x0 + pos*(x1-x0)

    ln, = ax.plot(
        [xx, xx],
        [y0+0.12, 6.95],
        color="#60899b",
        lw=0.75,
        ls="--",
        alpha=0,
        zorder=3
    )

    txt = ax.text(
        xx, 7.08,
        label,
        fontsize=12,
        weight="bold",
        color="white",
        ha="center",
        va="bottom",
        alpha=0,
        zorder=20
    )

    feature_lines.append(ln)
    feature_labels.append(txt)

# ============================================================
# H-alpha ZOOM / VELOCITY INFORMATION
# ============================================================

halpha_info = ax.text(
    8.0, 1.02,
    "Hα  •  BROAD P-CYGNI PROFILE",
    fontsize=13,
    weight="bold",
    color="white",
    ha="center",
    va="center",
    alpha=0,
    zorder=30
)

velocity_info = ax.text(
    8.0, 0.58,
    "FWHM ≈ 7,900 km/s     •     absorption blueshift ≈ 8,800 km/s",
    fontsize=10.5,
    color="#9fc6d7",
    ha="center",
    va="center",
    alpha=0,
    zorder=30
)

# ============================================================
# CLASSIFICATION
# ============================================================

classification = ax.text(
    8.0, 1.05,
    "HYDROGEN-RICH PHOTOSPHERE",
    fontsize=15,
    weight="bold",
    color="white",
    ha="center",
    alpha=0,
    zorder=35
)

classification2 = ax.text(
    8.0, 0.58,
    "TYPE II SUPERNOVA  •  z = 3.34",
    fontsize=13,
    color="#b8dce9",
    ha="center",
    alpha=0,
    zorder=35
)

# ============================================================
# FINAL CARD
# ============================================================

final_overlay = plt.Rectangle(
    (0,0), 16,9,
    facecolor="#03070d",
    edgecolor="none",
    alpha=0,
    zorder=50
)

ax.add_patch(final_overlay)

final_title = ax.text(
    8, 7.05,
    "SN HELIOS",
    fontsize=31,
    weight="bold",
    color="white",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_type = ax.text(
    8, 6.10,
    "TYPE II SUPERNOVA",
    fontsize=22,
    weight="bold",
    color="#dff5ff",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_z = ax.text(
    8, 5.42,
    "z = 3.34",
    fontsize=16,
    color="#9fc6d7",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_rule = ax.text(
    8, 4.25,
    "Hα   •   Hβ   •   He I",
    fontsize=19,
    weight="bold",
    color="white",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_halpha = ax.text(
    8, 3.22,
    "Hα FWHM ≈ 7,900 km/s",
    fontsize=14,
    color="#b5ced9",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_blue = ax.text(
    8, 2.62,
    "P-Cygni absorption blueshift ≈ 8,800 km/s",
    fontsize=12,
    color="#91aab5",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_message = ax.text(
    8, 1.55,
    "THE SPECTRUM REVEALS WHAT EXPLODED",
    fontsize=13,
    weight="bold",
    color="white",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

# ============================================================
# CLOCK
# ============================================================

clock_center = np.array([0.68, 0.58])
clock_r = 0.29

clock_circle = Circle(
    clock_center,
    clock_r,
    facecolor="none",
    edgecolor="#5e7681",
    linewidth=0.8,
    alpha=0.6,
    zorder=70
)

ax.add_patch(clock_circle)

for i in range(12):

    ang = 2*np.pi*i/12

    p1 = clock_center + 0.21*np.array([
        np.sin(ang), np.cos(ang)
    ])

    p2 = clock_center + 0.255*np.array([
        np.sin(ang), np.cos(ang)
    ])

    ax.plot(
        [p1[0], p2[0]],
        [p1[1], p2[1]],
        color="#5e7681",
        lw=0.7,
        alpha=0.6,
        zorder=70
    )

clock_hand, = ax.plot(
    [], [],
    color="#b8d5df",
    lw=1.2,
    alpha=0.8,
    zorder=71
)

# ============================================================
# ANIMATION
# ============================================================

def update(frame):

    t = frame / FPS

    # --------------------------------------------------------
    # CLOCK
    # --------------------------------------------------------

    theta = 2*np.pi*t/DURATION

    hand_end = clock_center + 0.205*np.array([
        np.sin(theta),
        np.cos(theta)
    ])

    clock_hand.set_data(
        [clock_center[0], hand_end[0]],
        [clock_center[1], hand_end[1]]
    )

    # --------------------------------------------------------
    # BACKGROUND
    # --------------------------------------------------------

    stars.set_alpha(
        0.18 * fade_in(t, 0.0, 0.8)
    )

    set_alpha(title, fade_in(t, 0.2, 0.7))
    set_alpha(subtitle, fade_in(t, 0.7, 0.7))

    # --------------------------------------------------------
    # 1–5 s — THE SOURCE
    # --------------------------------------------------------

    source_a = fade_window(t, 1.0, 5.7, 0.7)

    source_halo.set_alpha(0.18*source_a)
    source_core.set_alpha(0.95*source_a)

    # very subtle scintillation
    source_halo.set_radius(
        0.38 * (1 + 0.06*np.sin(t*5.0))
    )

    set_alpha(
        source_label,
        fade_window(t, 1.6, 5.4, 0.6)
    )

    set_alpha(
        source_info,
        fade_window(t, 2.0, 5.4, 0.6)
    )

    set_alpha(
        question,
        fade_window(t, 3.1, 5.5, 0.5)
    )

    # --------------------------------------------------------
    # 5–7 s — SPECTROGRAPH APPEARS
    # --------------------------------------------------------

    axes_a = fade_in(t, 5.2, 0.8)

    axis_x.set_alpha(0.75*axes_a)
    axis_y.set_alpha(0.75*axes_a)

    set_alpha(xlabel, axes_a)
    set_alpha(ylabel, axes_a)

    spectrum_ghost.set_alpha(
        0.14 * fade_in(t, 5.5, 0.8)
    )

    # --------------------------------------------------------
    # 6–9 s — SPECTRUM DRAWS ACROSS SCREEN
    # --------------------------------------------------------

    draw_fraction = smoothstep((t-5.8)/3.0)

    n = max(
        2,
        int(draw_fraction * len(spec_x))
    )

    spectrum_line.set_data(
        spec_x[:n],
        spec_y[:n]
    )

    spectrum_line.set_alpha(
        fade_in(t, 5.8, 0.4)
    )

    # --------------------------------------------------------
    # 8–12 s — IDENTIFY FEATURES
    # --------------------------------------------------------

    feature_times = [8.1, 9.0, 10.0]

    for i, start in enumerate(feature_times):

        a = fade_in(t, start, 0.55)

        feature_lines[i].set_alpha(
            0.55*a
        )

        feature_labels[i].set_alpha(a)

    # --------------------------------------------------------
    # 10.7–14 s — H-alpha physics
    # --------------------------------------------------------

    set_alpha(
        halpha_info,
        fade_window(t, 10.8, 14.8, 0.6)
    )

    set_alpha(
        velocity_info,
        fade_window(t, 11.4, 14.8, 0.6)
    )

    # Make H-alpha marker stronger during this stage

    if 10.8 <= t < 14.8:
        feature_lines[1].set_alpha(0.95)
        feature_labels[1].set_alpha(1.0)

    # --------------------------------------------------------
    # 13–16.5 s — CLASSIFICATION
    # --------------------------------------------------------

    set_alpha(
        classification,
        fade_window(t, 13.4, 16.6, 0.6)
    )

    set_alpha(
        classification2,
        fade_window(t, 13.9, 16.6, 0.6)
    )

    # --------------------------------------------------------
    # FINAL CARD
    #
    # Begins ~16.6 s.
    # About six seconds of stable reading time.
    # --------------------------------------------------------

    final_a = smoothstep((t-16.3)/0.7)

    final_overlay.set_alpha(
        0.97*final_a
    )

    set_alpha(
        final_title,
        fade_in(t, 16.55, 0.4)
    )

    set_alpha(
        final_type,
        fade_in(t, 16.8, 0.4)
    )

    set_alpha(
        final_z,
        fade_in(t, 17.0, 0.4)
    )

    set_alpha(
        final_rule,
        fade_in(t, 17.2, 0.4)
    )

    set_alpha(
        final_halpha,
        fade_in(t, 17.4, 0.4)
    )

    set_alpha(
        final_blue,
        fade_in(t, 17.6, 0.4)
    )

    set_alpha(
        final_message,
        fade_in(t, 17.8, 0.4)
    )

    return []

# ============================================================
# RENDER
# ============================================================

ani = animation.FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000/FPS,
    blit=False
)

plt.tight_layout(pad=0)

if OUTPUT.lower() == "webm":

    writer = animation.FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-deadline", "good",
            "-cpu-used", "2"
        ]
    )

    outfile = FILENAME + ".webm"

else:

    writer = animation.FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-preset", "medium"
        ]
    )

    outfile = FILENAME + ".mp4"

ani.save(
    outfile,
    writer=writer,
    dpi=DPI
)

plt.close(fig)

print(f"Saved: {outfile}")

В изображениях JWST SN Helios первоначально выглядит как слабый красный транзиент. Спектроскопия NIRSpec раскрывает его природу: в спектре обнаруживаются Hα, Hβ и He I. Широкая линия Hα имеет FWHM около 7900 км/с, а P-Cygni-поглощение смещено в синюю сторону примерно на 8800 км/с. Эти признаки показывают наличие быстро расширяющейся водородной фотосферы и позволяют классифицировать SN Helios как сверхновую II типа на красном смещении z = 3,34.

 In JWST imaging, SN Helios initially appears as a faint red transient. NIRSpec spectroscopy reveals its physical nature through Hα, Hβ, and He I features. The broad Hα line has an FWHM of about 7,900 km/s, while its P-Cygni absorption is blueshifted by about 8,800 km/s. These signatures reveal a rapidly expanding hydrogen-rich photosphere and identify SN Helios as a Type II supernova at redshift z = 3.34.

---

# Part 4. A SUPERNOVA DISGUISED AS A z≈16 GALAXY

In [ ]:
# A SUPERNOVA DISGUISED AS A z≈16 GALAXY
# SN Helios — photometric high-redshift impostor
#
# Self-contained Jupyter cell
# Based on Fujimoto et al. (2026), arXiv:2609.30440
#
# Output: WebM (VP9) or MP4 (H.264)
# Duration: 23 s @ 30 fps
#
# IMPORTANT:
# The plotted SED and spectrum are conceptual visualizations.
# Numerical annotations reproduce results reported in the paper.
# This is NOT a digitization of Figure 6.

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle, Rectangle

# ============================================================
# SETTINGS
# ============================================================

OUTPUT = "mp4"       # "webm" or "mp4"
FILENAME = "sn_helios_z16_impostor"

FPS = 30
DURATION = 23
NFRAMES = FPS * DURATION

FIGSIZE = (12.8, 7.2)
DPI = 120

# ============================================================
# HELPERS
# ============================================================

def clamp(x, a=0.0, b=1.0):
    return np.clip(x, a, b)

def smoothstep(x):
    x = clamp(x)
    return x*x*(3.0 - 2.0*x)

def fade_in(t, t0, duration=0.7):
    return smoothstep((t-t0)/duration)

def fade_window(t, t0, t1, fade=0.6):
    if t < t0 or t > t1:
        return 0.0
    return min(
        smoothstep((t-t0)/fade),
        smoothstep((t1-t)/fade)
    )

def set_alpha(obj, a):
    obj.set_alpha(clamp(a))

def gaussian(x, mu, sigma):
    return np.exp(-0.5*((x-mu)/sigma)**2)

# ============================================================
# FIGURE
# ============================================================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

fig.patch.set_facecolor("#03070d")
ax.set_facecolor("#03070d")

ax.set_xlim(0, 16)
ax.set_ylim(0, 9)
ax.axis("off")

# ============================================================
# STAR FIELD
# ============================================================

rng = np.random.default_rng(62)

sx = rng.uniform(0.1, 15.9, 220)
sy = rng.uniform(0.1, 8.9, 220)
ss = rng.uniform(0.7, 4.5, 220)

stars = ax.scatter(
    sx, sy,
    s=ss,
    c="white",
    linewidths=0,
    alpha=0,
    zorder=0
)

# ============================================================
# TITLE
# ============================================================

title = ax.text(
    0.65, 8.35,
    "A SUPERNOVA DISGUISED AS A z≈16 GALAXY",
    fontsize=23,
    weight="bold",
    color="white",
    ha="left",
    va="center",
    alpha=0,
    zorder=40
)

subtitle = ax.text(
    0.67, 7.92,
    "WHEN PHOTOMETRY ALONE TELLS THE WRONG STORY",
    fontsize=11.5,
    color="#9fc6d7",
    ha="left",
    va="center",
    alpha=0,
    zorder=40
)

# ============================================================
# OBJECT
# ============================================================

OBJ = np.array([3.0, 4.75])

obj_halo = Circle(
    OBJ,
    0.35,
    facecolor="#d4a890",
    edgecolor="none",
    alpha=0,
    zorder=8
)

obj_core = Circle(
    OBJ,
    0.075,
    facecolor="#fff1e7",
    edgecolor="none",
    alpha=0,
    zorder=9
)

ax.add_patch(obj_halo)
ax.add_patch(obj_core)

obj_label = ax.text(
    OBJ[0], 4.08,
    "FAINT RED SOURCE",
    fontsize=12,
    weight="bold",
    color="white",
    ha="center",
    alpha=0,
    zorder=20
)

# ============================================================
# PHOTOMETRIC FILTER DIAGRAM
# ============================================================

filter_names = [
    "F090W", "F115W", "F150W",
    "F200W", "F277W", "F356W", "F444W"
]

# Conceptual normalized fluxes:
# non-detections / very low blueward of the break,
# strong detection redward.
filter_flux = np.array([
    0.02, 0.025, 0.035,
    0.56, 0.82, 0.77, 0.68
])

fx = np.linspace(5.7, 14.3, len(filter_names))

filter_baseline = 2.75
filter_scale = 3.35

filter_stems = []
filter_points = []
filter_labels = []

for i, (name, flux) in enumerate(zip(filter_names, filter_flux)):

    yy = filter_baseline + flux*filter_scale

    stem, = ax.plot(
        [fx[i], fx[i]],
        [filter_baseline, yy],
        color="#63899a",
        lw=1.0,
        alpha=0,
        zorder=5
    )

    point, = ax.plot(
        [fx[i]], [yy],
        marker="o",
        markersize=6.5,
        markerfacecolor="white",
        markeredgecolor="#9fc6d7",
        markeredgewidth=1.0,
        alpha=0,
        zorder=8
    )

    label = ax.text(
        fx[i], 2.38,
        name,
        fontsize=8.5,
        color="#8099a4",
        ha="center",
        va="top",
        alpha=0,
        zorder=20
    )

    filter_stems.append(stem)
    filter_points.append(point)
    filter_labels.append(label)

baseline, = ax.plot(
    [5.35, 14.65],
    [filter_baseline, filter_baseline],
    color="#617681",
    lw=0.8,
    alpha=0,
    zorder=4
)

photometry_label = ax.text(
    10.0, 6.65,
    "JWST PHOTOMETRY",
    fontsize=12,
    weight="bold",
    color="white",
    ha="center",
    alpha=0,
    zorder=20
)

# ============================================================
# BREAK INDICATOR
# ============================================================

break_x = (fx[2] + fx[3]) / 2

break_line, = ax.plot(
    [break_x, break_x],
    [2.9, 6.25],
    color="#dceff7",
    lw=1.0,
    ls="--",
    alpha=0,
    zorder=6
)

break_text = ax.text(
    break_x, 6.32,
    "STRONG BREAK",
    fontsize=9.5,
    color="#dceff7",
    ha="center",
    va="bottom",
    alpha=0,
    zorder=20
)

break_value = ax.text(
    break_x, 5.83,
    "≳ 2 mag",
    fontsize=10,
    color="#9fc6d7",
    ha="center",
    alpha=0,
    zorder=20
)

# ============================================================
# PHOTOMETRIC INTERPRETATION
# ============================================================

photo_arrow = ax.text(
    8.0, 1.65,
    "PHOTOMETRY ONLY  →",
    fontsize=13,
    weight="bold",
    color="#9fc6d7",
    ha="right",
    va="center",
    alpha=0,
    zorder=30
)

photo_result = ax.text(
    8.15, 1.65,
    "z = 15.8",
    fontsize=25,
    weight="bold",
    color="white",
    ha="left",
    va="center",
    alpha=0,
    zorder=30
)

probability = ax.text(
    8.0, 1.03,
    "P(z > 10) > 0.9999",
    fontsize=12,
    color="#b6d2de",
    ha="center",
    va="center",
    alpha=0,
    zorder=30
)

# ============================================================
# TRANSITION / WARNING
# ============================================================

looks_convincing = ax.text(
    8.0, 4.6,
    "A CONVINCING HIGH-REDSHIFT GALAXY CANDIDATE",
    fontsize=17,
    weight="bold",
    color="white",
    ha="center",
    va="center",
    alpha=0,
    zorder=35
)

but_text = ax.text(
    8.0, 3.85,
    "BUT THE PHOTOMETRY IS NOT THE WHOLE STORY",
    fontsize=12,
    color="#9fc6d7",
    ha="center",
    va="center",
    alpha=0,
    zorder=35
)

# ============================================================
# SPECTROSCOPY PANEL
# ============================================================

spec_x = np.linspace(2.0, 14.0, 800)

u = np.linspace(0, 1, 800)

flux = (
    4.0
    + 0.10*np.sin(u*17)
    + 0.08*np.cos(u*31)
)

# conceptual features
flux += 0.35*gaussian(u, 0.30, 0.020)   # H-beta
flux += 1.00*gaussian(u, 0.59, 0.032)   # H-alpha emission
flux -= 0.38*gaussian(u, 0.54, 0.021)   # H-alpha absorption
flux += 0.42*gaussian(u, 0.85, 0.025)   # He I

spec_y = flux

spec_axis, = ax.plot(
    [2.0, 14.0],
    [3.3, 3.3],
    color="#647b86",
    lw=0.8,
    alpha=0,
    zorder=4
)

spec_line, = ax.plot(
    [],
    [],
    color="#dff5ff",
    lw=1.8,
    alpha=0,
    zorder=10
)

spec_label = ax.text(
    8.0, 6.55,
    "JWST / NIRSpec",
    fontsize=12,
    weight="bold",
    color="white",
    ha="center",
    alpha=0,
    zorder=20
)

# feature markers

feature_positions = [
    ("Hβ", 0.30),
    ("Hα", 0.59),
    ("He I", 0.85)
]

spec_feature_lines = []
spec_feature_labels = []

for name, pos in feature_positions:

    xx = 2.0 + pos*12.0

    ln, = ax.plot(
        [xx, xx],
        [3.35, 6.05],
        color="#64899a",
        lw=0.75,
        ls="--",
        alpha=0,
        zorder=5
    )

    txt = ax.text(
        xx, 6.10,
        name,
        fontsize=11,
        weight="bold",
        color="white",
        ha="center",
        va="bottom",
        alpha=0,
        zorder=20
    )

    spec_feature_lines.append(ln)
    spec_feature_labels.append(txt)

# ============================================================
# TRUE IDENTIFICATION
# ============================================================

truth_top = ax.text(
    8.0, 2.25,
    "SPECTROSCOPY  →  TYPE II SUPERNOVA",
    fontsize=16,
    weight="bold",
    color="white",
    ha="center",
    alpha=0,
    zorder=30
)

truth_z = ax.text(
    8.0, 1.65,
    "z = 3.34",
    fontsize=25,
    weight="bold",
    color="#dff5ff",
    ha="center",
    alpha=0,
    zorder=30
)

truth_name = ax.text(
    8.0, 1.05,
    "SN HELIOS",
    fontsize=12,
    color="#9fc6d7",
    ha="center",
    alpha=0,
    zorder=30
)

# ============================================================
# FINAL CARD
# ============================================================

final_overlay = Rectangle(
    (0, 0), 16, 9,
    facecolor="#03070d",
    edgecolor="none",
    alpha=0,
    zorder=50
)

ax.add_patch(final_overlay)

final_title = ax.text(
    8, 7.15,
    "SN HELIOS",
    fontsize=31,
    weight="bold",
    color="white",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_photo = ax.text(
    8, 5.85,
    "PHOTOMETRY ALONE",
    fontsize=14,
    color="#8faab6",
    ha="center",
    alpha=0,
    zorder=60
)

final_wrong = ax.text(
    8, 5.18,
    "z ≈ 16 GALAXY",
    fontsize=24,
    weight="bold",
    color="white",
    ha="center",
    alpha=0,
    zorder=60
)

final_vs = ax.text(
    8, 4.30,
    "↓",
    fontsize=22,
    color="#6f8b97",
    ha="center",
    alpha=0,
    zorder=60
)

final_spec = ax.text(
    8, 3.72,
    "SPECTROSCOPY",
    fontsize=14,
    color="#8faab6",
    ha="center",
    alpha=0,
    zorder=60
)

final_truth = ax.text(
    8, 3.00,
    "TYPE II SUPERNOVA  •  z = 3.34",
    fontsize=22,
    weight="bold",
    color="#dff5ff",
    ha="center",
    alpha=0,
    zorder=60
)

final_message = ax.text(
    8, 1.72,
    "A HIGH-REDSHIFT IMPOSTOR REVEALED BY ITS SPECTRUM",
    fontsize=12,
    weight="bold",
    color="white",
    ha="center",
    alpha=0,
    zorder=60
)

# ============================================================
# CLOCK
# ============================================================

clock_center = np.array([0.68, 0.58])
clock_r = 0.29

clock_circle = Circle(
    clock_center,
    clock_r,
    facecolor="none",
    edgecolor="#5e7681",
    linewidth=0.8,
    alpha=0.6,
    zorder=70
)

ax.add_patch(clock_circle)

for i in range(12):

    ang = 2*np.pi*i/12

    p1 = clock_center + 0.21*np.array([
        np.sin(ang), np.cos(ang)
    ])

    p2 = clock_center + 0.255*np.array([
        np.sin(ang), np.cos(ang)
    ])

    ax.plot(
        [p1[0], p2[0]],
        [p1[1], p2[1]],
        color="#5e7681",
        lw=0.7,
        alpha=0.6,
        zorder=70
    )

clock_hand, = ax.plot(
    [], [],
    color="#b8d5df",
    lw=1.2,
    alpha=0.8,
    zorder=71
)

# ============================================================
# ANIMATION
# ============================================================

def update(frame):

    t = frame / FPS

    # --------------------------------------------------------
    # CLOCK
    # --------------------------------------------------------

    theta = 2*np.pi*t/DURATION

    hand_end = clock_center + 0.205*np.array([
        np.sin(theta),
        np.cos(theta)
    ])

    clock_hand.set_data(
        [clock_center[0], hand_end[0]],
        [clock_center[1], hand_end[1]]
    )

    # --------------------------------------------------------
    # BACKGROUND + TITLE
    # --------------------------------------------------------

    stars.set_alpha(
        0.17 * fade_in(t, 0.0, 0.8)
    )

    set_alpha(title, fade_in(t, 0.2, 0.7))
    set_alpha(subtitle, fade_in(t, 0.8, 0.7))

    # --------------------------------------------------------
    # 1–4 s — SOURCE
    # --------------------------------------------------------

    oa = fade_window(t, 1.0, 7.8, 0.7)

    obj_halo.set_alpha(0.20*oa)
    obj_core.set_alpha(0.95*oa)

    obj_halo.set_radius(
        0.35*(1 + 0.05*np.sin(t*4.0))
    )

    set_alpha(
        obj_label,
        fade_window(t, 1.5, 7.5, 0.6)
    )

    # --------------------------------------------------------
    # 2–7.5 s — PHOTOMETRY
    # --------------------------------------------------------

    phot_a = fade_window(t, 2.0, 8.2, 0.7)

    baseline.set_alpha(0.65*phot_a)
    set_alpha(photometry_label, phot_a)

    for i in range(len(filter_names)):

        a = fade_in(t, 2.3 + i*0.18, 0.45)

        # also disappear with the panel
        a *= fade_window(t, 2.0, 8.2, 0.7)

        filter_stems[i].set_alpha(0.65*a)
        filter_points[i].set_alpha(a)
        filter_labels[i].set_alpha(a)

    # --------------------------------------------------------
    # BREAK
    # --------------------------------------------------------

    ba = fade_window(t, 4.0, 8.1, 0.6)

    break_line.set_alpha(0.8*ba)
    set_alpha(break_text, ba)
    set_alpha(break_value, ba)

    # --------------------------------------------------------
    # PHOTO-z RESULT
    # --------------------------------------------------------

    set_alpha(
        photo_arrow,
        fade_window(t, 5.0, 8.5, 0.6)
    )

    set_alpha(
        photo_result,
        fade_window(t, 5.3, 8.5, 0.6)
    )

    set_alpha(
        probability,
        fade_window(t, 5.8, 8.5, 0.6)
    )

    # --------------------------------------------------------
    # 8–10 s — THE TRAP
    # --------------------------------------------------------

    set_alpha(
        looks_convincing,
        fade_window(t, 8.0, 10.8, 0.5)
    )

    set_alpha(
        but_text,
        fade_window(t, 8.7, 10.8, 0.5)
    )

    # --------------------------------------------------------
    # 10–14 s — SPECTROSCOPY
    # --------------------------------------------------------

    spec_a = fade_window(t, 10.2, 16.7, 0.6)

    spec_axis.set_alpha(0.7*spec_a)
    set_alpha(spec_label, spec_a)

    draw = smoothstep((t-10.5)/2.6)

    n = max(2, int(draw*len(spec_x)))

    spec_line.set_data(
        spec_x[:n],
        spec_y[:n]
    )

    spec_line.set_alpha(spec_a)

    # feature labels

    starts = [11.8, 12.3, 12.8]

    for i, start in enumerate(starts):

        a = fade_in(t, start, 0.45)
        a *= fade_window(t, 10.2, 16.7, 0.6)

        spec_feature_lines[i].set_alpha(0.55*a)
        spec_feature_labels[i].set_alpha(a)

    # --------------------------------------------------------
    # 13–16.7 s — TRUE IDENTITY
    # --------------------------------------------------------

    set_alpha(
        truth_top,
        fade_window(t, 13.4, 16.8, 0.5)
    )

    set_alpha(
        truth_z,
        fade_window(t, 13.9, 16.8, 0.5)
    )

    set_alpha(
        truth_name,
        fade_window(t, 14.3, 16.8, 0.5)
    )

    # --------------------------------------------------------
    # FINAL CARD
    # roughly six seconds stable
    # --------------------------------------------------------

    fa = smoothstep((t-16.45)/0.65)

    final_overlay.set_alpha(0.97*fa)

    set_alpha(
        final_title,
        fade_in(t, 16.7, 0.4)
    )

    set_alpha(
        final_photo,
        fade_in(t, 16.95, 0.4)
    )

    set_alpha(
        final_wrong,
        fade_in(t, 17.15, 0.4)
    )

    set_alpha(
        final_vs,
        fade_in(t, 17.35, 0.4)
    )

    set_alpha(
        final_spec,
        fade_in(t, 17.50, 0.4)
    )

    set_alpha(
        final_truth,
        fade_in(t, 17.70, 0.4)
    )

    set_alpha(
        final_message,
        fade_in(t, 17.95, 0.4)
    )

    return []

# ============================================================
# RENDER
# ============================================================

ani = animation.FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000/FPS,
    blit=False
)

plt.tight_layout(pad=0)

if OUTPUT.lower() == "webm":

    writer = animation.FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-deadline", "good",
            "-cpu-used", "2"
        ]
    )

    outfile = FILENAME + ".webm"

else:

    writer = animation.FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-preset", "medium"
        ]
    )

    outfile = FILENAME + ".mp4"

ani.save(
    outfile,
    writer=writer,
    dpi=DPI
)

plt.close(fig)

print(f"Saved: {outfile}")

Если бы SN Helios наблюдали только фотометрически, её спектральное распределение энергии выглядело бы как чрезвычайно убедительный кандидат в галактики ранней Вселенной. Фотометрическая подгонка даёт z≈15,8 и вероятность z>10 выше 99,99%. Однако спектроскопия JWST/NIRSpec показывает линии водорода и гелия: объект на самом деле является сверхновой II типа при z=3,34. SN Helios демонстрирует, как транзиент может имитировать галактику экстремально высокого красного смещения. 2609.30440v1(1).pdf

With photometry alone, SN Helios would appear to be an exceptionally convincing candidate galaxy from the very early Universe. Photometric fitting yields z≈15.8 with P(z>10)>0.9999. JWST/NIRSpec spectroscopy, however, reveals hydrogen and helium features and shows that the source is actually a Type II supernova at z=3.34. SN Helios demonstrates how a transient can masquerade as an extreme-redshift galaxy. 2609.30440v1(1).pdf

---

# Part 5. MEASURING THE UNIVERSE WITH A REPEATING SUPERNOVA

In [ ]:
# MEASURING THE UNIVERSE WITH A REPEATING SUPERNOVA
# SN Helios — time-delay cosmography
#
# Self-contained Jupyter cell
# Based on Fujimoto et al. (2026), arXiv:2609.30440
#
# Output: WebM (VP9) or MP4 (H.264)
# Duration: 23 s @ 30 fps
#
# IMPORTANT:
# This is a conceptual explanation of time-delay cosmography.
# It does NOT calculate H0 from SN Helios.
# The future image has not yet been observed; lens models predict it.

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle, Ellipse, Rectangle
from matplotlib import patheffects

# ============================================================
# SETTINGS
# ============================================================

OUTPUT = "mp4"       # "webm" or "mp4"
FILENAME = "sn_helios_time_delay_cosmography"

FPS = 30
DURATION = 23
NFRAMES = FPS * DURATION

FIGSIZE = (12.8, 7.2)
DPI = 120

# ============================================================
# HELPERS
# ============================================================

def clamp(x, a=0.0, b=1.0):
    return np.clip(x, a, b)

def smoothstep(x):
    x = clamp(x)
    return x*x*(3.0 - 2.0*x)

def fade_in(t, t0, duration=0.7):
    return smoothstep((t-t0)/duration)

def fade_window(t, t0, t1, fade=0.6):
    if t < t0 or t > t1:
        return 0.0
    return min(
        smoothstep((t-t0)/fade),
        smoothstep((t1-t)/fade)
    )

def set_alpha(obj, a):
    obj.set_alpha(clamp(a))

def bezier(p0, p1, p2, p3, n=350):
    u = np.linspace(0, 1, n)
    return (
        ((1-u)**3)[:,None]*p0 +
        (3*(1-u)**2*u)[:,None]*p1 +
        (3*(1-u)*u**2)[:,None]*p2 +
        (u**3)[:,None]*p3
    )

def moving_segment(points, fraction, width=0.04):
    fraction = clamp(fraction)

    n = len(points)
    c = int(fraction*(n-1))
    half = max(3, int(width*n))

    lo = max(0, c-half)
    hi = min(n, c+half+1)

    return points[lo:hi]

# ============================================================
# FIGURE
# ============================================================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

fig.patch.set_facecolor("#03070d")
ax.set_facecolor("#03070d")

ax.set_xlim(0, 16)
ax.set_ylim(0, 9)
ax.set_aspect("equal")
ax.axis("off")

# ============================================================
# BACKGROUND
# ============================================================

rng = np.random.default_rng(93)

sx = rng.uniform(0.1, 15.9, 230)
sy = rng.uniform(0.1, 8.9, 230)
ss = rng.uniform(0.7, 4.5, 230)

stars = ax.scatter(
    sx, sy,
    s=ss,
    c="white",
    linewidths=0,
    alpha=0,
    zorder=0
)

# ============================================================
# TITLE
# ============================================================

title = ax.text(
    0.65, 8.35,
    "MEASURING THE UNIVERSE WITH A REPEATING SUPERNOVA",
    fontsize=21.5,
    weight="bold",
    color="white",
    ha="left",
    va="center",
    alpha=0,
    zorder=40
)

subtitle = ax.text(
    0.67, 7.92,
    "SN HELIOS  •  TIME-DELAY COSMOGRAPHY",
    fontsize=11.5,
    color="#9fc6d7",
    ha="left",
    va="center",
    alpha=0,
    zorder=40
)

# ============================================================
# SOURCE / LENS / OBSERVER
# ============================================================

SN = np.array([2.0, 4.8])
LENS = np.array([7.25, 4.8])
OBS = np.array([13.75, 4.8])

sn_halo = Circle(
    SN, 0.30,
    facecolor="#c9eaff",
    edgecolor="none",
    alpha=0,
    zorder=8
)

sn_core = Circle(
    SN, 0.07,
    facecolor="white",
    edgecolor="none",
    alpha=0,
    zorder=9
)

ax.add_patch(sn_halo)
ax.add_patch(sn_core)

sn_label = ax.text(
    SN[0], 4.20,
    "SN HELIOS\nz = 3.34",
    fontsize=10,
    color="white",
    ha="center",
    va="top",
    linespacing=1.3,
    alpha=0,
    zorder=30
)

# lens

lens_outer = Ellipse(
    LENS,
    2.2, 2.8,
    facecolor="#173448",
    edgecolor="#507e91",
    linewidth=0.9,
    alpha=0,
    zorder=2
)

lens_mass = Ellipse(
    LENS,
    3.0, 3.7,
    facecolor="none",
    edgecolor="#385e6e",
    linewidth=0.7,
    alpha=0,
    zorder=1
)

ax.add_patch(lens_outer)
ax.add_patch(lens_mass)

offsets = np.array([
    [0.00, 0.00],
    [-0.55, 0.46],
    [0.57, 0.40],
    [-0.60,-0.44],
    [0.52,-0.53],
    [0.08, 0.86],
    [-0.10,-0.86]
])

galaxies = []

for i, off in enumerate(offsets):

    g = Ellipse(
        LENS + off,
        0.48 if i == 0 else 0.29,
        0.23 if i == 0 else 0.14,
        angle=(i*39)%170,
        facecolor="#d5e3e8",
        edgecolor="none",
        alpha=0,
        zorder=6
    )

    ax.add_patch(g)
    galaxies.append(g)

lens_label = ax.text(
    LENS[0], 2.87,
    "GALAXY CLUSTER\nLENS MODEL",
    fontsize=10,
    color="#9fc6d7",
    ha="center",
    va="top",
    linespacing=1.3,
    alpha=0,
    zorder=30
)

# observer

obs_outer = Circle(
    OBS, 0.24,
    facecolor="none",
    edgecolor="#a5c9d7",
    linewidth=1.4,
    alpha=0,
    zorder=8
)

obs_inner = Circle(
    OBS, 0.06,
    facecolor="white",
    edgecolor="none",
    alpha=0,
    zorder=9
)

ax.add_patch(obs_outer)
ax.add_patch(obs_inner)

obs_label = ax.text(
    OBS[0], 4.22,
    "OBSERVER",
    fontsize=10,
    color="#9fc6d7",
    ha="center",
    va="top",
    alpha=0,
    zorder=30
)

# ============================================================
# TWO LIGHT PATHS
# ============================================================

path1 = bezier(
    SN,
    np.array([4.7, 5.15]),
    np.array([9.0, 5.60]),
    OBS
)

path2 = bezier(
    SN,
    np.array([4.6, 3.70]),
    np.array([9.1, 2.60]),
    OBS
)

paths = [path1, path2]

path_lines = []
pulse_lines = []

for pts in paths:

    line, = ax.plot(
        pts[:,0], pts[:,1],
        color="#63abc5",
        lw=1.2,
        alpha=0,
        zorder=4
    )

    pulse, = ax.plot(
        [], [],
        color="white",
        lw=3.0,
        alpha=0,
        solid_capstyle="round",
        zorder=10
    )

    pulse.set_path_effects([
        patheffects.Stroke(
            linewidth=7,
            foreground="#65b9d7",
            alpha=0.18
        ),
        patheffects.Normal()
    ])

    path_lines.append(line)
    pulse_lines.append(pulse)

# ============================================================
# DETECTION EVENTS / TIMELINE
# ============================================================

timeline_y = 1.55
timeline_x0 = 3.2
timeline_x1 = 12.8

timeline, = ax.plot(
    [timeline_x0, timeline_x1],
    [timeline_y, timeline_y],
    color="#647b86",
    lw=1.0,
    alpha=0,
    zorder=10
)

tick1, = ax.plot(
    [5.0, 5.0],
    [1.36, 1.74],
    color="white",
    lw=1.5,
    alpha=0,
    zorder=11
)

tick2, = ax.plot(
    [11.0, 11.0],
    [1.36, 1.74],
    color="white",
    lw=1.5,
    alpha=0,
    zorder=11
)

image1 = ax.text(
    5.0, 2.02,
    "IMAGE 1",
    fontsize=11,
    weight="bold",
    color="white",
    ha="center",
    alpha=0,
    zorder=20
)

image2 = ax.text(
    11.0, 2.02,
    "FUTURE IMAGE",
    fontsize=11,
    weight="bold",
    color="white",
    ha="center",
    alpha=0,
    zorder=20
)

now_label = ax.text(
    5.0, 1.08,
    "observed",
    fontsize=9.5,
    color="#8faab6",
    ha="center",
    alpha=0,
    zorder=20
)

future_label = ax.text(
    11.0, 1.08,
    "predicted",
    fontsize=9.5,
    color="#8faab6",
    ha="center",
    alpha=0,
    zorder=20
)

delta_t = ax.text(
    8.0, 1.72,
    "Δt",
    fontsize=19,
    weight="bold",
    color="#dff5ff",
    ha="center",
    alpha=0,
    zorder=20
)

delta_years = ax.text(
    8.0, 1.08,
    "≈ 2–6 years",
    fontsize=11,
    color="#a9c9d8",
    ha="center",
    alpha=0,
    zorder=20
)

# ============================================================
# COSMOGRAPHY EQUATION FLOW
# ============================================================

measure_delay = ax.text(
    3.0, 5.70,
    "MEASURE\nTIME DELAY",
    fontsize=13,
    weight="bold",
    color="white",
    ha="center",
    va="center",
    linespacing=1.3,
    alpha=0,
    zorder=35
)

plus1 = ax.text(
    5.1, 5.70,
    "+",
    fontsize=24,
    color="#718b97",
    ha="center",
    va="center",
    alpha=0,
    zorder=35
)

model_lens = ax.text(
    7.2, 5.70,
    "MODEL THE\nGRAVITATIONAL LENS",
    fontsize=13,
    weight="bold",
    color="white",
    ha="center",
    va="center",
    linespacing=1.3,
    alpha=0,
    zorder=35
)

plus2 = ax.text(
    9.45, 5.70,
    "+",
    fontsize=24,
    color="#718b97",
    ha="center",
    va="center",
    alpha=0,
    zorder=35
)

geometry = ax.text(
    11.35, 5.70,
    "SOURCE + LENS\nGEOMETRY",
    fontsize=13,
    weight="bold",
    color="white",
    ha="center",
    va="center",
    linespacing=1.3,
    alpha=0,
    zorder=35
)

arrow_result = ax.text(
    13.2, 5.70,
    "→",
    fontsize=26,
    color="#8eacba",
    ha="center",
    va="center",
    alpha=0,
    zorder=35
)

h0_result = ax.text(
    14.45, 5.70,
    "H₀",
    fontsize=29,
    weight="bold",
    color="#dff5ff",
    ha="center",
    va="center",
    alpha=0,
    zorder=35
)

h0_caption = ax.text(
    14.45, 5.08,
    "EXPANSION RATE",
    fontsize=8.5,
    color="#8faab6",
    ha="center",
    va="center",
    alpha=0,
    zorder=35
)

# ============================================================
# CONCEPT TEXT
# ============================================================

concept1 = ax.text(
    8.0, 7.10,
    "THE DELAY IS NOT JUST A CURIOSITY",
    fontsize=17,
    weight="bold",
    color="white",
    ha="center",
    alpha=0,
    zorder=35
)

concept2 = ax.text(
    8.0, 6.58,
    "IT ENCODES COSMOLOGICAL DISTANCE INFORMATION",
    fontsize=11,
    color="#9fc6d7",
    ha="center",
    alpha=0,
    zorder=35
)

# ============================================================
# FINAL CARD
# ============================================================

final_overlay = Rectangle(
    (0, 0), 16, 9,
    facecolor="#03070d",
    edgecolor="none",
    alpha=0,
    zorder=50
)

ax.add_patch(final_overlay)

final_title = ax.text(
    8, 7.15,
    "SN HELIOS",
    fontsize=31,
    weight="bold",
    color="white",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_subtitle = ax.text(
    8, 6.28,
    "A SUPERNOVA WITH A COSMIC CLOCK",
    fontsize=15,
    color="#9fc6d7",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_delay = ax.text(
    8, 5.15,
    "TIME DELAY  Δt",
    fontsize=21,
    weight="bold",
    color="white",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_plus = ax.text(
    8, 4.42,
    "+",
    fontsize=23,
    color="#708b97",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_lens = ax.text(
    8, 3.75,
    "GRAVITATIONAL-LENS MODEL",
    fontsize=18,
    weight="bold",
    color="white",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_arrow = ax.text(
    8, 3.02,
    "↓",
    fontsize=22,
    color="#708b97",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_h0 = ax.text(
    8, 2.30,
    "H₀",
    fontsize=34,
    weight="bold",
    color="#dff5ff",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

final_message = ax.text(
    8, 1.50,
    "TIME-DELAY COSMOGRAPHY BEYOND REDSHIFT 3",
    fontsize=12,
    weight="bold",
    color="#9fc6d7",
    ha="center",
    va="center",
    alpha=0,
    zorder=60
)

# ============================================================
# CLOCK
# ============================================================

clock_center = np.array([0.68, 0.58])
clock_r = 0.29

clock_circle = Circle(
    clock_center,
    clock_r,
    facecolor="none",
    edgecolor="#5e7681",
    linewidth=0.8,
    alpha=0.6,
    zorder=70
)

ax.add_patch(clock_circle)

for i in range(12):

    ang = 2*np.pi*i/12

    p1 = clock_center + 0.21*np.array([
        np.sin(ang), np.cos(ang)
    ])

    p2 = clock_center + 0.255*np.array([
        np.sin(ang), np.cos(ang)
    ])

    ax.plot(
        [p1[0], p2[0]],
        [p1[1], p2[1]],
        color="#5e7681",
        lw=0.7,
        alpha=0.6,
        zorder=70
    )

clock_hand, = ax.plot(
    [], [],
    color="#b8d5df",
    lw=1.2,
    alpha=0.8,
    zorder=71
)

# ============================================================
# ANIMATION
# ============================================================

def update(frame):

    t = frame / FPS

    # --------------------------------------------------------
    # CLOCK
    # --------------------------------------------------------

    theta = 2*np.pi*t/DURATION

    hand_end = clock_center + 0.205*np.array([
        np.sin(theta),
        np.cos(theta)
    ])

    clock_hand.set_data(
        [clock_center[0], hand_end[0]],
        [clock_center[1], hand_end[1]]
    )

    # --------------------------------------------------------
    # BACKGROUND / TITLE
    # --------------------------------------------------------

    stars.set_alpha(
        0.17 * fade_in(t, 0.0, 0.8)
    )

    set_alpha(title, fade_in(t, 0.2, 0.7))
    set_alpha(subtitle, fade_in(t, 0.8, 0.7))

    # --------------------------------------------------------
    # 1–7 s — LENS SYSTEM
    # --------------------------------------------------------

    scene_a = fade_window(t, 1.0, 9.0, 0.7)

    sn_halo.set_alpha(0.20*scene_a)
    sn_core.set_alpha(0.95*scene_a)

    sn_halo.set_radius(
        0.30*(1 + 0.05*np.sin(t*4.5))
    )

    set_alpha(sn_label, scene_a)

    lens_outer.set_alpha(0.16*scene_a)
    lens_mass.set_alpha(0.28*scene_a)

    for i, g in enumerate(galaxies):
        g.set_alpha(
            (0.78 if i == 0 else 0.48)*scene_a
        )

    set_alpha(lens_label, scene_a)

    obs_outer.set_alpha(0.82*scene_a)
    obs_inner.set_alpha(0.95*scene_a)
    set_alpha(obs_label, scene_a)

    path_a = fade_window(t, 2.2, 9.0, 0.7)

    for line in path_lines:
        line.set_alpha(0.38*path_a)

    # --------------------------------------------------------
    # LIGHT PULSES
    # --------------------------------------------------------

    timings = [
        (2.8, 5.5),
        (2.8, 7.8)
    ]

    for i, (ts, te) in enumerate(timings):

        frac = (t-ts)/(te-ts)

        if 0 <= frac <= 1:

            seg = moving_segment(paths[i], frac)

            pulse_lines[i].set_data(
                seg[:,0],
                seg[:,1]
            )

            pulse_lines[i].set_alpha(0.98)

        else:

            pulse_lines[i].set_data([], [])
            pulse_lines[i].set_alpha(0)

    # --------------------------------------------------------
    # 4–9 s — TIMELINE / Δt
    # --------------------------------------------------------

    time_a = fade_window(t, 4.2, 10.2, 0.7)

    timeline.set_alpha(0.75*time_a)

    tick1.set_alpha(time_a)
    image1.set_alpha(time_a)
    now_label.set_alpha(time_a)

    # Future image appears later in the explanation

    second_a = fade_window(t, 7.0, 10.2, 0.6)

    tick2.set_alpha(second_a)
    image2.set_alpha(second_a)
    future_label.set_alpha(second_a)

    set_alpha(
        delta_t,
        fade_window(t, 7.4, 10.2, 0.5)
    )

    set_alpha(
        delta_years,
        fade_window(t, 7.7, 10.2, 0.5)
    )

    # --------------------------------------------------------
    # 9–11 s — KEY IDEA
    # --------------------------------------------------------

    set_alpha(
        concept1,
        fade_window(t, 9.0, 12.0, 0.5)
    )

    set_alpha(
        concept2,
        fade_window(t, 9.5, 12.0, 0.5)
    )

    # --------------------------------------------------------
    # 11–16.7 s — COSMOGRAPHY PIPELINE
    # --------------------------------------------------------

    set_alpha(
        measure_delay,
        fade_window(t, 10.8, 16.8, 0.5)
    )

    set_alpha(
        plus1,
        fade_window(t, 11.4, 16.8, 0.5)
    )

    set_alpha(
        model_lens,
        fade_window(t, 11.8, 16.8, 0.5)
    )

    set_alpha(
        plus2,
        fade_window(t, 12.5, 16.8, 0.5)
    )

    set_alpha(
        geometry,
        fade_window(t, 12.9, 16.8, 0.5)
    )

    set_alpha(
        arrow_result,
        fade_window(t, 13.6, 16.8, 0.5)
    )

    set_alpha(
        h0_result,
        fade_window(t, 14.0, 16.8, 0.5)
    )

    set_alpha(
        h0_caption,
        fade_window(t, 14.2, 16.8, 0.5)
    )

    # --------------------------------------------------------
    # FINAL CARD
    # Starts ~16.5 s, then remains readable for ~6 s.
    # --------------------------------------------------------

    fa = smoothstep((t-16.35)/0.65)

    final_overlay.set_alpha(0.97*fa)

    set_alpha(
        final_title,
        fade_in(t, 16.60, 0.4)
    )

    set_alpha(
        final_subtitle,
        fade_in(t, 16.85, 0.4)
    )

    set_alpha(
        final_delay,
        fade_in(t, 17.10, 0.4)
    )

    set_alpha(
        final_plus,
        fade_in(t, 17.30, 0.4)
    )

    set_alpha(
        final_lens,
        fade_in(t, 17.48, 0.4)
    )

    set_alpha(
        final_arrow,
        fade_in(t, 17.65, 0.4)
    )

    set_alpha(
        final_h0,
        fade_in(t, 17.82, 0.4)
    )

    set_alpha(
        final_message,
        fade_in(t, 18.00, 0.4)
    )

    return []

# ============================================================
# RENDER
# ============================================================

ani = animation.FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000/FPS,
    blit=False
)

plt.tight_layout(pad=0)

if OUTPUT.lower() == "webm":

    writer = animation.FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-deadline", "good",
            "-cpu-used", "2"
        ]
    )

    outfile = FILENAME + ".webm"

else:

    writer = animation.FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-preset", "medium"
        ]
    )

    outfile = FILENAME + ".mp4"

ani.save(
    outfile,
    writer=writer,
    dpi=DPI
)

plt.close(fig)

print(f"Saved: {outfile}")

Гравитационная линза превращает SN Helios в естественный космический эксперимент. Различные изображения одной сверхновой проходят разные оптические пути и появляются с временной задержкой. Если будущее изображение будет зарегистрировано, измеренную задержку можно объединить с моделью распределения массы скопления и геометрией системы. Это основа космографии по временным задержкам — метода, позволяющего получать информацию о космологических расстояниях и постоянной Хаббла H₀. SN Helios открывает возможность применить этот подход к сверхновой на красном смещении выше 3.

 Gravitational lensing turns SN Helios into a natural cosmological experiment. Different images of the same supernova follow different optical paths and therefore appear with a time delay. If the predicted future image is observed, its measured delay can be combined with the cluster mass model and the geometry of the system. This is the basis of time-delay cosmography, which provides information about cosmological distances and the Hubble constant H₀. SN Helios opens the possibility of applying this method to a supernova source beyond redshift 3.